### Bilaplaciano
Resolvemos el problema para el bilaplaciano,

$$
\left\{\begin{array}{l}
 -\Delta^2 u = f \quad\text{en }\Omega \\
u = g_1 \quad\text{sobre }\partial\Omega \\
\Delta u = g_2 \quad\text{sobre }\partial\Omega 
\end{array}\right.
$$
con $\Omega = (0,1)$, $f=-\pi^3(\pi x \sin(\pi x) - 4\cos(\pi x))$,  $g_1(0)=g_1(1)=0$, $g_2(0)=2\pi$, $g_2(1)= -2\pi$. Solución exacta: $x\sin(\pi x)$.

Haciendo el cambio $v = \Delta u$
resulta:
$$
\begin{array}{l}
\left\{\begin{array}{ll}
-\Delta v = f \quad\text{en }\Omega \\
v = g_2 \quad\text{sobre }\partial\Omega 
\end{array}\right. 
\\
\left\{ \begin{array}{ll}
- \Delta u +v = 0  \quad\text{en }\Omega \\
u = g_1 \quad\text{sobre }\partial\Omega 
\end{array}\right. \\
\end{array}
$$
que podemos resolver consecutivamente.

In [13]:
%reset -f
import mfem.ser as mfem
import numpy as np
from glvis import glvis

### Malla

In [25]:
mesh = mfem.Mesh(100)
dim = mesh.Dimension()
mesh.bdr_attributes.ToList()

[1, 2]

#### Espacios

In [26]:
order = 1
h2 = mfem.H1_FECollection(order, dim)
U_space = mfem.FiniteElementSpace(mesh, h2)

### Condición frontera

In [27]:
bd = mfem.intArray([1,1])
# Asignamos etiquetas al espacio de elementos 
boundary_dofs = mfem.intArray()
U_space.GetBoundaryTrueDofs(boundary_dofs)

#### Condiciones frontera

In [28]:
ux = mfem.GridFunction(U_space)
vx = mfem.GridFunction(U_space)

f1 = mfem.PWConstCoefficient(mfem.Vector([0.,0.]))
f2 = mfem.PWConstCoefficient(mfem.Vector([2*np.pi,-2.*np.pi]))
ux.ProjectBdrCoefficient(f1,bd)
vx.ProjectBdrCoefficient(f2,bd)

### Formulación variacional
$$\begin{array}{c}
\displaystyle \int_\Omega \nabla v\cdot \nabla w  = \int_\Omega fw \\
\displaystyle \int_\Omega \nabla u\cdot\nabla z = - \int_\Omega vz   \end{array}$$
con $u \in g_1 + H^1_0(\Omega)$, $v\in g_2 + H^1_0(\Omega)$, $w,z\in H_1^0(\Omega)$.

El problema está desacoplado, por lo que resolvemos primero en $v$ y luego en $u$.

In [29]:
class ffunc(mfem.PyCoefficient):
    def EvalValue(self,x):
        return -np.pi**3*(np.pi*x[0]*np.sin(np.pi*x[0]) - 4*np.cos(np.pi*x[0]))
fcoef = ffunc()

#### Segun miembro problema para `v`

In [30]:
gform = mfem.LinearForm(U_space)
gform.AddDomainIntegrator(mfem.DomainLFIntegrator(fcoef))
gform.Assemble()

In [31]:
cVarf = mfem.BilinearForm(U_space)
cVarf.AddDomainIntegrator(mfem.DiffusionIntegrator())
cVarf.Assemble()

In [32]:
A = mfem.SparseMatrix()
B = mfem.Vector()
V = mfem.Vector()
cVarf.FormLinearSystem(boundary_dofs,vx,gform,A,V,B)
mfem.CG(A, B, V, 0, 200, 1e-12, 0.0)
cVarf.RecoverFEMSolution(V, gform, vx)

#### Segundo miembro problema para `u`
Para generar un coeficiente a partir de una `GridFunction` usamos `GridFunctionCoefficient`

In [33]:
vcoef = mfem.GridFunctionCoefficient(vx)

fform = mfem.LinearForm(U_space)
fform.AddDomainIntegrator(mfem.DomainLFIntegrator(mfem.ProductCoefficient(-1.,vcoef)))
fform.Assemble()

No es necesario calcular otra vez la matriz de rigidez, pues es la misma. El método `FormLinearSystem` detecta si la matriz `A` es válida, y en tal caso, solo modifica el segundo miembro.

In [34]:
U = mfem.Vector()
cVarf.FormLinearSystem(boundary_dofs,ux,fform,A,U,B)
mfem.CG(A, B, U, 0, 200, 1e-12, 0.0)
cVarf.RecoverFEMSolution(U, fform, ux)

#### Cálculo del error

In [35]:
# Solución exacta
class usol(mfem.PyCoefficient):
    def EvalValue(self,x):
        return x[0]*np.sin(np.pi*x[0])
        
class vsol(mfem.PyCoefficient):
    def EvalValue(self,x):
        return np.pi*(-np.pi*x[0]*np.sin(np.pi*x[0]) + 2*np.cos(np.pi*x[0]))

uc = usol()
vc = vsol()

irs = [mfem.IntRules.Get(i,2) for i in range(mfem.Geometry.NumGeom)]


err_u = ux.ComputeL2Error(uc, irs)
err_v = vx.ComputeL2Error(vc, irs)
print(err_u)
print(err_v)


8.647479481437613e-05
0.0008721966950567104


In [36]:
glvis((mesh,ux), keys="cRR")